In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

In [3]:
INPUT_ROOT = Path("/kaggle/input/datasets/helinatefera/freight-rate")

train_path = next(INPUT_ROOT.rglob("train-test.csv"))
validation_path = next(INPUT_ROOT.rglob("validation.csv"))

train_raw = pd.read_csv(train_path)
validation_raw = pd.read_csv(validation_path)

print("Training shape:", train_raw.shape)
print("Validation shape:", validation_raw.shape)

print("Duplicate training rows:", train_raw.duplicated().sum())
print("Duplicate validation rows:", validation_raw.duplicated().sum())

print("Duplicate training load IDs:",
      train_raw["load_id"].duplicated().sum())

print("Duplicate validation load IDs:",
      validation_raw["load_id"].duplicated().sum())

Training shape: (48000, 14)
Validation shape: (12000, 13)
Duplicate training rows: 0
Duplicate validation rows: 0
Duplicate training load IDs: 0
Duplicate validation load IDs: 0


In [4]:
CATEGORICAL_COLUMNS = [
    "pickup",
    "delivery",
    "equipment"
]

BASE_NUMERIC_COLUMNS = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "weight",
    "market_index",
    "quote_signal"
]

COORDINATE_COLUMNS = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon"
]

ROUTE_IMPUTE_COLUMNS = [
    "weight",
    "distance",
    "quote_signal"
]

TARGET_COLUMN = "posted_rate"


def normalize_text(series):
    return (
        series
        .astype("string")
        .str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def clean_types(df):
    result = df.copy()

    # Clean categorical columns while preserving missing values
    for column in CATEGORICAL_COLUMNS:
        original_missing = result[column].isna()

        result[column] = normalize_text(result[column])

        result[f"{column}_missing"] = (
            original_missing.astype("int8")
        )

        result[column] = result[column].fillna("__MISSING__")

    # Parse date
    original_date = result["date"].copy()

    result["date"] = pd.to_datetime(
        original_date,
        errors="coerce"
    )

    result["date_parse_error"] = (
        original_date.notna()
        & result["date"].isna()
    ).astype("int8")

    # Parse numerical columns
    for column in BASE_NUMERIC_COLUMNS:
        original_values = result[column].copy()

        converted = pd.to_numeric(
            original_values,
            errors="coerce"
        )

        result[f"{column}_parse_error"] = (
            original_values.notna()
            & converted.isna()
        ).astype("int8")

        result[column] = converted

    # Invalid positive-only variables
    for column in ["weight", "distance", "market_index"]:
        invalid = (
            result[column].notna()
            & (result[column] <= 0)
        )

        result[f"{column}_invalid"] = (
            invalid.astype("int8")
        )

        result[f"{column}_missing_or_invalid"] = (
            result[column].isna() | invalid
        ).astype("int8")

        result.loc[invalid, column] = np.nan

    # Coordinate validation
    coordinate_ranges = {
        "pickup_lat": (-90, 90),
        "pickup_lon": (-180, 180),
        "delivery_lat": (-90, 90),
        "delivery_lon": (-180, 180),
    }

    for column, (minimum, maximum) in coordinate_ranges.items():
        invalid = (
            result[column].notna()
            & ~result[column].between(minimum, maximum)
        )

        result[f"{column}_invalid"] = (
            invalid.astype("int8")
        )

        result.loc[invalid, column] = np.nan

    return result

In [5]:
def create_quality_report(df):
    cleaned = clean_types(df)

    flag_columns = [
        column
        for column in cleaned.columns
        if (
            "_missing" in column
            or "_invalid" in column
            or "_problem" in column
            or "_parse_error" in column
        )
    ]

    report = pd.DataFrame({
        "flag": flag_columns,
        "rows_affected": [
            int(cleaned[column].sum())
            for column in flag_columns
        ]
    })

    report["percent_of_rows"] = (
        report["rows_affected"] / len(df) * 100
    )

    return report.sort_values(
        "rows_affected",
        ascending=False
    )


train_quality = create_quality_report(train_raw)
validation_quality = create_quality_report(validation_raw)

display(train_quality)
display(validation_quality)

print(
    "Missing target values:",
    train_raw[TARGET_COLUMN].isna().sum()
)

print(
    "Non-positive target values:",
    (
        pd.to_numeric(
            train_raw[TARGET_COLUMN],
            errors="coerce"
        ) <= 0
    ).sum()
)

,flag,rows_affected,percent_of_rows
13,weight_missing_or_invalid,592,1.233333
17,market_index_missing_or_invalid,374,0.779167
12,weight_invalid,292,0.608333
2,equipment_missing,0,0.000000
0,pickup_missing,0,0.000000
1,delivery_missing,0,0.000000
5,pickup_lon_parse_error,0,0.000000
4,pickup_lat_parse_error,0,0.000000
3,date_parse_error,0,0.000000
6,delivery_lat_parse_error,0,0.000000


,flag,rows_affected,percent_of_rows
13,weight_missing_or_invalid,310,2.583333
17,market_index_missing_or_invalid,249,2.075000
12,weight_invalid,145,1.208333
2,equipment_missing,0,0.000000
0,pickup_missing,0,0.000000
1,delivery_missing,0,0.000000
5,pickup_lon_parse_error,0,0.000000
4,pickup_lat_parse_error,0,0.000000
3,date_parse_error,0,0.000000
6,delivery_lat_parse_error,0,0.000000


Missing target values: 0
Non-positive target values: 0


In [6]:
train_dates = pd.to_datetime(
    train_raw["date"],
    errors="coerce"
)

cutoff_date = pd.Timestamp("2025-10-01")

fold_train_raw = train_raw.loc[
    train_dates < cutoff_date
].copy()

fold_valid_raw = train_raw.loc[
    train_dates >= cutoff_date
].copy()

print("Model training rows:", len(fold_train_raw))
print("Holdout rows:", len(fold_valid_raw))

print(
    "Training date range:",
    fold_train_raw["date"].min(),
    "to",
    fold_train_raw["date"].max()
)

print(
    "Holdout date range:",
    fold_valid_raw["date"].min(),
    "to",
    fold_valid_raw["date"].max()
)

Model training rows: 43147
Holdout rows: 4853
Training date range: 2025-01-01 to 2025-09-30
Holdout date range: 2025-10-01 to 2025-10-31


In [7]:
def make_route_key(df):
    return (
        df["pickup"].astype("string")
        + "||"
        + df["delivery"].astype("string")
    )


def fit_artifacts(clean_training):
    training = clean_training.copy()
    training["_route_key"] = make_route_key(training)
    training["_date_key"] = training["date"].dt.normalize()

    artifacts = {}

    # Global medians
    artifacts["global_medians"] = (
        training[BASE_NUMERIC_COLUMNS].median()
    )

    artifacts["global_coordinates"] = (
        training[COORDINATE_COLUMNS].median()
    )

    # Route-level medians
    artifacts["route_medians"] = {}

    for column in ROUTE_IMPUTE_COLUMNS:
        artifacts["route_medians"][column] = (
            training
            .groupby("_route_key")[column]
            .median()
            .to_dict()
        )

    # Equipment-level medians
    artifacts["equipment_medians"] = {}

    for column in ROUTE_IMPUTE_COLUMNS:
        artifacts["equipment_medians"][column] = (
            training
            .groupby("equipment")[column]
            .median()
            .to_dict()
        )

    # Market-index date and equipment medians
    artifacts["market_date_medians"] = (
        training
        .groupby("_date_key")["market_index"]
        .median()
        .to_dict()
    )

    artifacts["market_equipment_medians"] = (
        training
        .groupby("equipment")["market_index"]
        .median()
        .to_dict()
    )

    # Location coordinate medians
    artifacts["location_medians"] = {
        "pickup": {
            "pickup_lat": (
                training.groupby("pickup")["pickup_lat"]
                .median()
                .to_dict()
            ),
            "pickup_lon": (
                training.groupby("pickup")["pickup_lon"]
                .median()
                .to_dict()
            )
        },
        "delivery": {
            "delivery_lat": (
                training.groupby("delivery")["delivery_lat"]
                .median()
                .to_dict()
            ),
            "delivery_lon": (
                training.groupby("delivery")["delivery_lon"]
                .median()
                .to_dict()
            )
        }
    }

    # Frequency features
    artifacts["frequencies"] = {
        "route": training["_route_key"]
        .value_counts()
        .to_dict(),

        "pickup": training["pickup"]
        .value_counts()
        .to_dict(),

        "delivery": training["delivery"]
        .value_counts()
        .to_dict(),

        "equipment": training["equipment"]
        .value_counts()
        .to_dict()
    }

    artifacts["train_start"] = training["date"].min()

    return artifacts

In [8]:
fold_train_clean = clean_types(fold_train_raw)

fold_artifacts = fit_artifacts(
    fold_train_clean
)

In [10]:
def add_geo_features(df):
    result = df.copy()

    lat1 = np.radians(result["pickup_lat"])
    lon1 = np.radians(result["pickup_lon"])
    lat2 = np.radians(result["delivery_lat"])
    lon2 = np.radians(result["delivery_lon"])

    latitude_difference = lat2 - lat1
    longitude_difference = lon2 - lon1

    haversine_a = (
        np.sin(latitude_difference / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(longitude_difference / 2) ** 2
    )

    result["geo_distance_miles"] = (
        2
        * 3958.7613
        * np.arcsin(
            np.sqrt(np.clip(haversine_a, 0, 1))
        )
    )

    result["latitude_difference"] = (
        result["delivery_lat"]
        - result["pickup_lat"]
    )

    result["longitude_difference"] = (
        result["delivery_lon"]
        - result["pickup_lon"]
    )

    result["absolute_latitude_difference"] = (
        result["latitude_difference"].abs()
    )

    result["absolute_longitude_difference"] = (
        result["longitude_difference"].abs()
    )

    coordinates_available = (
        result[COORDINATE_COLUMNS]
        .notna()
        .all(axis=1)
    )

    result["same_coordinates"] = (
        coordinates_available
        & (
            result["pickup_lat"]
            == result["delivery_lat"]
        )
        & (
            result["pickup_lon"]
            == result["delivery_lon"]
        )
    ).astype("int8")

    result["distance_geo_ratio_miles"] = (
        result["distance"]
        / result["geo_distance_miles"].replace(
            0,
            np.nan
        )
    )

    return result

In [11]:
def transform_data(
    df,
    artifacts,
    use_geo_distance_fallback=False
):
    result = clean_types(df)

    # Remove target if it exists
    result = result.drop(
        columns=[TARGET_COLUMN],
        errors="ignore"
    )

    # Coordinate fallback from known location medians
    coordinates_missing = (
        result[COORDINATE_COLUMNS]
        .isna()
        .any(axis=1)
    )

    coordinate_mapping = {
        "pickup": {
            "pickup_lat": "pickup_lat",
            "pickup_lon": "pickup_lon"
        },
        "delivery": {
            "delivery_lat": "delivery_lat",
            "delivery_lon": "delivery_lon"
        }
    }

    for location_type, mapping in coordinate_mapping.items():
        for coordinate, source_column in mapping.items():
            location_values = (
                result[location_type]
                .map(
                    artifacts["location_medians"]
                    [location_type]
                    [source_column]
                )
            )

            result[source_column] = (
                result[source_column]
                .fillna(location_values)
                .fillna(
                    artifacts["global_coordinates"]
                    [source_column]
                )
            )

    result["coordinates_imputed"] = (
        coordinates_missing.astype("int8")
    )

    # Geographic features
    result = add_geo_features(result)

    result["distance_geo_fallback_available"] = (
        result["distance"].isna()
        & result["geo_distance_miles"].notna()
    ).astype("int8")

    # Use this only after confirming distance is in miles
    if use_geo_distance_fallback:
        result["distance"] = (
            result["distance"]
            .fillna(result["geo_distance_miles"])
        )

    route = make_route_key(result)

    # Hierarchical imputation
    for column in ROUTE_IMPUTE_COLUMNS:
        result[column] = (
            result[column]
            .fillna(
                route.map(
                    artifacts["route_medians"][column]
                )
            )
            .fillna(
                result["equipment"].map(
                    artifacts["equipment_medians"][column]
                )
            )
            .fillna(
                artifacts["global_medians"][column]
            )
        )

    # Market index: date -> equipment -> global median
    date_key = result["date"].dt.normalize()

    result["market_index"] = (
        result["market_index"]
        .fillna(
            date_key.map(
                artifacts["market_date_medians"]
            )
        )
        .fillna(
            result["equipment"].map(
                artifacts["market_equipment_medians"]
            )
        )
        .fillna(
            artifacts["global_medians"]["market_index"]
        )
    )

    # Final safety fallback for base numeric columns
    for column in BASE_NUMERIC_COLUMNS:
        result[column] = result[column].fillna(
            artifacts["global_medians"][column]
        )

    # Route features
    result["route"] = route

    result["reverse_route"] = (
        result["delivery"].astype("string")
        + "||"
        + result["pickup"].astype("string")
    )

    result["route_frequency"] = (
        route
        .map(artifacts["frequencies"]["route"])
        .fillna(0)
        .astype("int32")
    )

    result["pickup_frequency"] = (
        result["pickup"]
        .map(artifacts["frequencies"]["pickup"])
        .fillna(0)
        .astype("int32")
    )

    result["delivery_frequency"] = (
        result["delivery"]
        .map(artifacts["frequencies"]["delivery"])
        .fillna(0)
        .astype("int32")
    )

    # Time features
    date_values = result["date"]
    month = date_values.dt.month
    day_of_week = date_values.dt.dayofweek

    result["year"] = (
        date_values.dt.year.fillna(-1)
    )

    result["month"] = month.fillna(-1)
    result["day_of_month"] = (
        date_values.dt.day.fillna(-1)
    )
    result["day_of_week"] = (
        day_of_week.fillna(-1)
    )
    result["day_of_year"] = (
        date_values.dt.dayofyear.fillna(-1)
    )

    result["is_weekend"] = (
        day_of_week.ge(5)
        .fillna(False)
        .astype("int8")
    )

    result["month_sin"] = (
        np.sin(2 * np.pi * month / 12)
        .fillna(0)
    )

    result["month_cos"] = (
        np.cos(2 * np.pi * month / 12)
        .fillna(0)
    )

    result["day_of_week_sin"] = (
        np.sin(2 * np.pi * day_of_week / 7)
        .fillna(0)
    )

    result["day_of_week_cos"] = (
        np.cos(2 * np.pi * day_of_week / 7)
        .fillna(0)
    )

    result["days_since_training_start"] = (
        date_values - artifacts["train_start"]
    ).dt.days.fillna(-1)

    # Stable numerical transformations
    result["distance_log1p"] = np.log1p(
        result["distance"].clip(lower=0)
    )

    result["weight_log1p"] = np.log1p(
        result["weight"].clip(lower=0)
    )

    result["market_index_log1p"] = np.log1p(
        result["market_index"].clip(lower=0)
    )

    result["weight_per_mile"] = (
        result["weight"]
        / result["distance"].replace(0, np.nan)
    )

    result["weight_per_mile_log1p"] = np.log1p(
        result["weight_per_mile"].clip(lower=0)
    )

    # Useful interactions
    result["market_distance_interaction"] = (
        result["market_index"]
        * result["distance"]
    )

    result["quote_distance_interaction"] = (
        result["quote_signal"]
        * result["distance"]
    )

    # Date is represented by derived features
    result = result.drop(
        columns=["date", "load_id"],
        errors="ignore"
    )

    return result

In [12]:
X_fold_train = transform_data(
    fold_train_raw,
    fold_artifacts
)

X_fold_valid = transform_data(
    fold_valid_raw,
    fold_artifacts
)

y_fold_train = pd.to_numeric(
    fold_train_raw[TARGET_COLUMN],
    errors="coerce"
)

y_fold_valid = pd.to_numeric(
    fold_valid_raw[TARGET_COLUMN],
    errors="coerce"
)

X_fold_train = X_fold_train.reset_index(drop=True)
X_fold_valid = X_fold_valid.reset_index(drop=True)

y_fold_train = y_fold_train.reset_index(drop=True)
y_fold_valid = y_fold_valid.reset_index(drop=True)

print("Processed training shape:", X_fold_train.shape)
print("Processed holdout shape:", X_fold_valid.shape)

print(
    "Training target missing:",
    y_fold_train.isna().sum()
)

print(
    "Remaining numeric missing values:",
    X_fold_train.select_dtypes(
        include=np.number
    ).isna().sum().sum()
)

Processed training shape: (43147, 65)
Processed holdout shape: (4853, 65)
Training target missing: 0
Remaining numeric missing values: 0


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

MODEL_CATEGORICAL_COLUMNS = [
    "pickup",
    "delivery",
    "equipment",
    "route",
    "reverse_route"
]

MODEL_NUMERIC_COLUMNS = [
    column
    for column in X_fold_train.columns
    if column not in MODEL_CATEGORICAL_COLUMNS
]

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        MODEL_NUMERIC_COLUMNS
    ),
    (
        "categorical",
        categorical_pipeline,
        MODEL_CATEGORICAL_COLUMNS
    )
])

X_train_encoded = preprocessor.fit_transform(
    X_fold_train
)

X_valid_encoded = preprocessor.transform(
    X_fold_valid
)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded holdout shape:", X_valid_encoded.shape)

Encoded training shape: (43147, 8207)
Encoded holdout shape: (4853, 8207)


In [14]:
def rolling_month_folds(
    df,
    minimum_training_months=5
):
    dates = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    periods = dates.dt.to_period("M")
    months = sorted(periods.dropna().unique())

    for index in range(
        minimum_training_months,
        len(months)
    ):
        validation_month = months[index]

        train_mask = periods < validation_month
        validation_mask = periods == validation_month

        yield (
            validation_month,
            train_mask,
            validation_mask
        )


for (
    validation_month,
    train_mask,
    validation_mask
) in rolling_month_folds(train_raw):

    fold_train = train_raw.loc[train_mask].copy()
    fold_valid = train_raw.loc[validation_mask].copy()

    fold_clean = clean_types(fold_train)
    fold_artifacts = fit_artifacts(fold_clean)

    X_train = transform_data(
        fold_train,
        fold_artifacts
    )

    X_valid = transform_data(
        fold_valid,
        fold_artifacts
    )

    print(
        validation_month,
        "training rows:",
        len(X_train),
        "validation rows:",
        len(X_valid)
    )

2025-06 training rows: 24023 validation rows: 4783
2025-07 training rows: 28806 validation rows: 4912
2025-08 training rows: 33718 validation rows: 4759
2025-09 training rows: 38477 validation rows: 4670
2025-10 training rows: 43147 validation rows: 4853


In [15]:
full_clean = clean_types(train_raw)

full_artifacts = fit_artifacts(
    full_clean
)

X_full_train = transform_data(
    train_raw,
    full_artifacts
)

X_official_validation = transform_data(
    validation_raw,
    full_artifacts
)

y_full = pd.to_numeric(
    train_raw[TARGET_COLUMN],
    errors="coerce"
).reset_index(drop=True)

X_full_train = X_full_train.reset_index(drop=True)
X_official_validation = (
    X_official_validation.reset_index(drop=True)
)

print("Final training features:", X_full_train.shape)
print(
    "Official validation features:",
    X_official_validation.shape
)

Final training features: (48000, 65)
Official validation features: (12000, 65)


In [16]:
final_preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        MODEL_NUMERIC_COLUMNS
    ),
    (
        "categorical",
        categorical_pipeline,
        MODEL_CATEGORICAL_COLUMNS
    )
])

X_full_encoded = final_preprocessor.fit_transform(
    X_full_train
)

X_official_encoded = final_preprocessor.transform(
    X_official_validation
)

print("Final encoded training:", X_full_encoded.shape)
print("Final encoded validation:", X_official_encoded.shape)

Final encoded training: (48000, 8219)
Final encoded validation: (12000, 8219)


In [17]:
OUTPUT_DIR = Path("/kaggle/working/processed")
OUTPUT_DIR.mkdir(exist_ok=True)

X_full_train.to_csv(
    OUTPUT_DIR / "processed_train_features.csv",
    index=False
)

X_official_validation.to_csv(
    OUTPUT_DIR / "processed_validation_features.csv",
    index=False
)

pd.DataFrame({
    "load_id": validation_raw["load_id"]
}).to_csv(
    OUTPUT_DIR / "validation_ids.csv",
    index=False
)

print("Saved processed files to:", OUTPUT_DIR)

Saved processed files to: /kaggle/working/processed


# Data Preprocessing Summary

An advanced, leakage safe preprocessing pipeline was developed for freight rate prediction. The process included data inspection, type normalization, invalid value detection, missing value handling, geographic consistency checks, route and time feature engineering, training only imputation, categorical encoding, rolling time based validation, and final preparation of future validation data. Raw data was preserved, negative weights were treated as invalid, valid negative longitude values were retained, and all transformations were designed to produce reliable, consistent, model ready data.